## 1. Bronze Adverse Events Schema

**Question**

What is the structure and data type definition of the Bronze safety adverse events dataset?

**Purpose**

Understand the raw Bronze adverse event structure before designing Silver normalization, type handling, data-quality rules, and ingestion processing.

In [0]:
%sql
DESCRIBE TABLE clinical_trial_intelligence.bronze.safety_adverse_events;

### Result

The Bronze adverse events table contains an event identifier (`ae_id`), subject and study identifiers, site identifier, and clinical event attributes including adverse event term, MedDRA System Organ Class, severity, seriousness flag, onset and resolution dates, outcome, causality assessment, and action taken. Source lineage metadata is also present.

Key date fields (`onset_date`, `resolution_date`) are stored as `DATE`. All categorical fields are stored as `STRING`, confirming that normalization and standardization are required at the Silver layer.

### Design conclusion

The Bronze schema provides the required attributes for Silver adverse event processing. The Silver pipeline will standardize categorical values including `severity` and `serious_flag`, apply clinical data-quality rules, and preserve source lineage for auditability.

## 2. Sample Bronze Adverse Event Records

**Question**

What does an actual Bronze adverse event record look like, including clinical attributes and ingestion lineage metadata?

**Purpose**

Inspect representative source records to understand the values available for Silver transformation, data-quality validation, and categorical standardization.

In [0]:
%sql

SELECT
    ae_id,
    subject_id,
    study_id,
    site_id,
    ae_term,
    meddra_soc,
    severity,
    serious_flag,
    onset_date,
    resolution_date,
    outcome,
    related_to_study_drug,
    action_taken,
    _source_file_name,
    _source_file_modification_ts,
    _ingestion_ts,
    _ingestion_date
FROM clinical_trial_intelligence.bronze.safety_adverse_events
LIMIT 20;

### Result

The Bronze sample contains adverse event identifiers, subject and study assignments, clinical event attributes, causality and action fields, and ingestion lineage metadata.

The sample reveals source-data quality and standardization issues. Severity is represented using multiple raw formats such as `MILD`, `Moderate`, `mild`, `1`, and `Grade 1`. The `serious_flag` field uses a mix of `N`, `Y`, `YES`, `1`, and `TRUE`. These inconsistencies require normalization and reference-driven standardization before the Silver layer.

### Design conclusion

The Bronze sample confirms that the adverse event domain requires categorical standardization at Silver for `severity` and `serious_flag`. The Silver pipeline will normalize these fields using reference mappings and apply clinical data-quality rules.

## 3. Adverse Event Source File Pattern

**Question**

Do the adverse event source files represent a complete snapshot of all events or an incremental delta of new events?

**Purpose**

Compare record volumes and key uniqueness across source files to determine the correct ingestion strategy for the Silver adverse event pipeline.

In [0]:
%sql
SELECT
    _source_file_name,
    COUNT(*) AS row_count,
    COUNT(DISTINCT ae_id) AS distinct_ae_ids
FROM clinical_trial_intelligence.bronze.safety_adverse_events
GROUP BY _source_file_name
ORDER BY _source_file_name;

### Result

The source-file inventory shows a clear difference between the initial adverse event file and subsequent files.

`adverse_events_20260825.csv` contains 1,699 rows representing 1,699 distinct AE identifiers. Each subsequent source file contains a smaller number of records, ranging from 11 to 77 rows, all with fully distinct AE identifiers.

### Design conclusion

The incremental pattern confirms that each successive file introduces new adverse event records rather than re-publishing previously reported events. This is consistent with an append-only ingestion pattern.

The Silver adverse event pipeline can therefore use a Streaming Table with a standard append flow, without the need for CDC processing. The `ae_id` column will serve as the primary key for data-quality validation.

## 4. AEs Appearing Across Multiple Source Files

**Question**

Do any individual adverse event identifiers appear in more than one EDC source file?

**Purpose**

Confirm that AE records are never re-issued across source files, validating the append-only ingestion assumption before finalizing the Silver pipeline architecture.

In [0]:
%sql

SELECT
    ae_id,
    COUNT(*) AS record_count,
    COUNT(DISTINCT _source_file_name) AS source_file_count,
    MIN(_source_file_name) AS first_source_file,
    MAX(_source_file_name) AS latest_source_file,
    COLLECT_SET(_source_file_name) AS source_files
FROM clinical_trial_intelligence.bronze.safety_adverse_events
WHERE ae_id IS NOT NULL
  AND TRIM(ae_id) <> ''
GROUP BY ae_id
HAVING COUNT(*) > 1
ORDER BY record_count DESC, ae_id
LIMIT 20;

### Result

No adverse event identifiers appear in more than one source file. The query returns zero rows.

Every `ae_id` in the Bronze dataset is unique to a single source file, confirming that the source never re-emits a previously reported adverse event record.

### Design conclusion

The Bronze adverse event data follows a strict append-only pattern. No CDC processing is required at the Silver layer.

The Silver adverse event pipeline will use a Streaming Table ingesting new records from each incremental Bronze file. A data-quality rule will enforce `ae_id` non-null uniqueness as a defensive control.

## 5. AE Key Null and Blank Analysis

**Question**

Are there any adverse event records with a missing or blank `ae_id` in the Bronze dataset?

**Purpose**

Determine whether records without a valid AE identifier exist before finalizing the data-quality rules for the Silver adverse event pipeline.

In [0]:
%sql

SELECT
    _source_file_name,

    COUNT(*) AS total_rows,

    COUNT_IF(ae_id IS NULL) AS null_ae_ids,

    COUNT_IF(
        ae_id IS NOT NULL
        AND TRIM(ae_id) = ''
    ) AS blank_ae_ids,

    COUNT_IF(
        ae_id IS NOT NULL
        AND TRIM(ae_id) <> ''
    ) AS populated_ae_ids,

    COUNT(
        DISTINCT CASE
            WHEN ae_id IS NOT NULL
             AND TRIM(ae_id) <> ''
            THEN TRIM(ae_id)
        END
    ) AS distinct_populated_ae_ids

FROM clinical_trial_intelligence.bronze.safety_adverse_events

GROUP BY _source_file_name

ORDER BY _source_file_name;

### Result

Across all 10 source files, every record contains a populated and distinct `ae_id`. There are zero NULL and zero blank AE identifiers in any source file.

All 1,952 Bronze adverse event records have a valid primary key.

### Design conclusion

`ae_id` is a reliable primary key for Silver adverse event data-quality validation.

Although no missing AE identifiers are present in the current Bronze data, a `missing_ae_id` data-quality rule will be included in the Silver pipeline as a defensive control for future source files.

## 6. Source Filename Date Parsing

**Question**

Can a valid logical source date be derived from every Bronze adverse event source filename?

**Purpose**

Verify that the `YYYYMMDD` component of `_source_file_name` can be parsed reliably before using the derived date as an ingestion audit attribute in the Silver adverse event pipeline.

In [0]:
%sql

SELECT
    COUNT(*) AS total_rows,

    COUNT_IF(
        TO_DATE(
            REGEXP_EXTRACT(
                _source_file_name,
                'adverse_events_(\\d{8})\\.csv',
                1
            ),
            'yyyyMMdd'
        ) IS NULL
    ) AS invalid_source_dates,

    COUNT(DISTINCT _source_file_name) AS source_file_count,

    MIN(
        TO_DATE(
            REGEXP_EXTRACT(
                _source_file_name,
                'adverse_events_(\\d{8})\\.csv',
                1
            ),
            'yyyyMMdd'
        )
    ) AS earliest_source_date,

    MAX(
        TO_DATE(
            REGEXP_EXTRACT(
                _source_file_name,
                'adverse_events_(\\d{8})\\.csv',
                1
            ),
            'yyyyMMdd'
        )
    ) AS latest_source_date

FROM clinical_trial_intelligence.bronze.safety_adverse_events;

### Result

The Bronze adverse event dataset contains 1,952 records across 10 source files, covering source dates from 2026-08-25 through 2026-09-07.

All 1,952 records successfully produce a valid date from the `YYYYMMDD` component of `_source_file_name`. The number of records with an invalid derived source date is 0.

### Design conclusion

The source filename provides a reliable source-date value for every Bronze adverse event record.

`source_snapshot_date` can therefore be safely derived from `_source_file_name` and used as an ingestion audit attribute and optional partition column in the Silver adverse event table.

## 7. Severity Reference Mapping Coverage

**Question**

Can the distinct severity values present in the Bronze adverse event data be consistently standardized using `ref_severity`?

**Purpose**

Compare normalized Bronze severity values against the Silver severity reference table to identify unmapped values and confirm that reference-driven standardization is appropriate for the adverse event pipeline.

In [0]:
%sql

WITH bronze_severity AS (

    SELECT
        UPPER(TRIM(severity)) AS normalized_severity,
        COUNT(*) AS bronze_record_count

    FROM clinical_trial_intelligence.bronze.safety_adverse_events

    WHERE severity IS NOT NULL
      AND TRIM(severity) <> ''

    GROUP BY
        UPPER(TRIM(severity))
),

severity_reference AS (

    SELECT
        UPPER(TRIM(raw_severity)) AS normalized_raw_severity,
        UPPER(TRIM(standard_severity)) AS standard_severity

    FROM clinical_trial_intelligence.silver.ref_severity

    WHERE raw_severity IS NOT NULL
      AND TRIM(raw_severity) <> ''
)

SELECT
    b.normalized_severity AS bronze_severity_value,
    b.bronze_record_count,
    COLLECT_SET(r.standard_severity) AS mapped_standard_values,
    COUNT(DISTINCT r.standard_severity) AS standard_value_count,

    CASE
        WHEN COUNT(r.normalized_raw_severity) = 0
            THEN 'UNMAPPED'
        WHEN COUNT(DISTINCT r.standard_severity) > 1
            THEN 'AMBIGUOUS'
        ELSE 'MAPPED'
    END AS mapping_status

FROM bronze_severity b

LEFT JOIN severity_reference r
    ON b.normalized_severity = r.normalized_raw_severity

GROUP BY
    b.normalized_severity,
    b.bronze_record_count

ORDER BY
    mapping_status DESC,
    b.normalized_severity;

### Result

The Bronze adverse event dataset contains distinct severity representations spanning multiple encoding conventions.

Nine values are successfully resolved through `ref_severity`: `1` → `MILD`, `2` → `MODERATE`, `3` → `SEVERE`, `GRADE 1` → `MILD`, `GRADE 2` → `MODERATE`, `GRADE 3` → `SEVERE`, `MILD` → `MILD`, `MODERATE` → `MODERATE`, and `SEVERE` → `SEVERE`.

Four values are unmapped: `GRADE 4`, `N/A`, `UNKNOWN`, and records where `severity` is NULL.

### Design conclusion

Reference-driven severity standardization is suitable for the majority of Bronze adverse event records.

The pipeline will normalize the source severity value using `UPPER(TRIM(...))` and resolve it through `ref_severity`.

The standardized Silver severity domain is therefore:

- `MILD`
- `MODERATE`
- `SEVERE`

An `unmappable_severity` data-quality rule will warn on records where `severity` cannot be resolved through the reference table. Records with `GRADE 4`, `N/A`, `UNKNOWN`, or NULL severity will be flagged but retained.

## 8. Serious Flag Standardization

**Question**

What distinct values does the `serious_flag` field contain in the Bronze adverse event dataset, and can they be consistently normalized to a binary indicator?

**Purpose**

Assess the standardization requirements for the seriousness indicator before defining the Silver normalization expression.

In [0]:
%sql

SELECT
    serious_flag                     AS raw_serious_flag,
    UPPER(TRIM(serious_flag))        AS normalized_serious_flag,
    COUNT(*)                         AS record_count,

    CASE
        WHEN UPPER(TRIM(serious_flag)) IN ('Y', 'YES', '1', 'TRUE')
            THEN 'YES'
        WHEN UPPER(TRIM(serious_flag)) IN ('N', 'NO', '0', 'FALSE')
            THEN 'NO'
        ELSE 'UNMAPPABLE'
    END AS standardized_value

FROM clinical_trial_intelligence.bronze.safety_adverse_events

GROUP BY
    serious_flag,
    UPPER(TRIM(serious_flag))

ORDER BY
    standardized_value,
    record_count DESC;

### Result

The Bronze adverse event dataset contains five distinct `serious_flag` values: `N` (1,783 records), `Y` (162 records), `YES` (3 records), `1` (2 records), and `TRUE` (2 records).

All five values can be unambiguously mapped to a binary `YES`/`NO` indicator:

- `Y`, `YES`, `1`, `TRUE` → `YES`
- `N` → `NO`

No NULL, blank, or ambiguous values are present in the current source data.

### Design conclusion

The Silver adverse event pipeline will normalize `serious_flag` using an inline `CASE WHEN` expression on `UPPER(TRIM(serious_flag))`.

The standardized Silver domain is:

- `YES` — the event meets the regulatory seriousness criteria
- `NO` — the event is non-serious

No reference table is required for this field. A `missing_serious_flag` data-quality rule will be added as a defensive control for future NULL values.

## 9. Dataset Profile and Completeness

### Question

What is the overall volume and completeness profile of the Bronze adverse-events feed?

### Purpose

Establish the Bronze completeness baseline before defining Silver data-quality rules.

This identifies missing business keys, subject/study/site relationships, core adverse-event attributes, clinical dates, severity and seriousness information, outcomes, causality fields, rescued records, and the number of source files contributing to the dataset.

In [0]:
%sql

SELECT
    COUNT(*) AS total_rows,

    COUNT(DISTINCT ae_id) AS distinct_ae_ids,

    COUNT_IF(
        ae_id IS NULL OR TRIM(ae_id) = ''
    ) AS missing_ae_id,

    COUNT_IF(
        subject_id IS NULL OR TRIM(subject_id) = ''
    ) AS missing_subject_id,

    COUNT_IF(
        study_id IS NULL OR TRIM(study_id) = ''
    ) AS missing_study_id,

    COUNT_IF(
        site_id IS NULL OR TRIM(site_id) = ''
    ) AS missing_site_id,

    COUNT_IF(
        ae_term IS NULL OR TRIM(ae_term) = ''
    ) AS missing_ae_term,

    COUNT_IF(
        meddra_soc IS NULL OR TRIM(meddra_soc) = ''
    ) AS missing_meddra_soc,

    COUNT_IF(
        severity IS NULL OR TRIM(severity) = ''
    ) AS missing_severity,

    COUNT_IF(
        serious_flag IS NULL OR TRIM(serious_flag) = ''
    ) AS missing_serious_flag,

    COUNT_IF(
        onset_date IS NULL
    ) AS missing_onset_date,

    COUNT_IF(
        outcome IS NULL OR TRIM(outcome) = ''
    ) AS missing_outcome,

    COUNT_IF(
        related_to_study_drug IS NULL
        OR TRIM(related_to_study_drug) = ''
    ) AS missing_related_to_study_drug,

    COUNT_IF(
        action_taken IS NULL OR TRIM(action_taken) = ''
    ) AS missing_action_taken,

    COUNT_IF(
        _rescued_data IS NOT NULL
    ) AS rescued_data_rows,

    COUNT(DISTINCT _source_file_name)
        AS source_file_count

FROM clinical_trial_intelligence.bronze.safety_adverse_events;

### Result

The Bronze adverse-events feed contains 1,952 records across 10 source files, with all 1,952 `ae_id` values distinct.

The dataset is highly complete overall. Missingness is limited to three fields:

- `subject_id`: 11 records
- `severity`: 6 records
- `onset_date`: 26 records

No missing values were identified for `ae_id`, `study_id`, `site_id`, `ae_term`, `meddra_soc`, `serious_flag`, `outcome`, `related_to_study_drug`, or `action_taken`.

No rescued-data records were observed.

The completeness profile therefore indicates that Silver DQ investigation should focus primarily on subject linkage, severity completeness, and adverse-event onset dates.

### Conclusion

The Bronze adverse-events feed has a stable business key and strong overall completeness.

The 11 records without `subject_id` cannot be reliably associated with a trial participant and therefore require further DQ evaluation.

The 6 missing severity values and 26 missing onset dates also require business-context analysis before determining whether they should be quarantined or retained with a non-blocking DQ indicator.

No general completeness remediation is required for the remaining evaluated fields.

## 10. Subject, Study, and Site Referential Integrity

### Question

Do adverse-event records resolve to valid trial subjects, studies, and sites, and are their study/site relationships consistent?

### Purpose

Determine whether each adverse event can be reliably associated with the clinical-trial entities required for downstream safety analysis.

An adverse event may contain populated identifiers while still referencing an unknown subject, study, or site, or a subject/site belonging to a different study.

These conditions must be distinguished because they represent different data-quality failures and remediation paths.

In [0]:
%sql

WITH ae AS (

    SELECT
        ae_id,
        NULLIF(TRIM(subject_id), '') AS subject_id,
        NULLIF(TRIM(study_id), '') AS study_id,
        NULLIF(TRIM(site_id), '') AS site_id

    FROM clinical_trial_intelligence.bronze.safety_adverse_events
),

subjects AS (

    SELECT DISTINCT
        subject_id,
        study_id,
        site_id

    FROM clinical_trial_intelligence.bronze.edc_subjects

    WHERE subject_id IS NOT NULL
),

studies AS (

    SELECT DISTINCT
        study_id

    FROM clinical_trial_intelligence.bronze.ctms_studies

    WHERE study_id IS NOT NULL
),

sites AS (

    SELECT DISTINCT
        site_id,
        study_id

    FROM clinical_trial_intelligence.bronze.ctms_sites

    WHERE site_id IS NOT NULL
)

SELECT

    COUNT(*) AS total_rows,

    COUNT_IF(
        ae.subject_id IS NULL
    ) AS missing_subject_id,

    COUNT_IF(
        ae.subject_id IS NOT NULL
        AND s.subject_id IS NULL
    ) AS unknown_subject_id,

    COUNT_IF(
        ae.study_id IS NOT NULL
        AND st.study_id IS NULL
    ) AS unknown_study_id,

    COUNT_IF(
        ae.site_id IS NOT NULL
        AND si.site_id IS NULL
    ) AS unknown_site_id,

    COUNT_IF(
        s.subject_id IS NOT NULL
        AND ae.study_id <> s.study_id
    ) AS subject_study_mismatch,

    COUNT_IF(
        s.subject_id IS NOT NULL
        AND ae.site_id <> s.site_id
    ) AS subject_site_mismatch,

    COUNT_IF(
        si.site_id IS NOT NULL
        AND ae.study_id <> si.study_id
    ) AS site_study_mismatch

FROM ae

LEFT JOIN subjects s
    ON ae.subject_id = s.subject_id

LEFT JOIN studies st
    ON ae.study_id = st.study_id

LEFT JOIN sites si
    ON ae.site_id = si.site_id;

### Result

Among 1,952 Bronze adverse-event records:

- 11 records have no `subject_id`.
- 13 records contain a populated `subject_id` that does not resolve to the Bronze subject population.
- No unknown `study_id` values were identified.
- No unknown `site_id` values were identified.
- No subject-study mismatches were identified.
- 14 records contain a site assignment inconsistent with the referenced subject.
- No site-study mismatches were identified.

The results show that study and site master references are complete, while subject linkage and subject-site consistency contain identifiable data-quality defects.

### Conclusion

Adverse-event referential integrity cannot be assumed solely because study and site identifiers are populated.

Silver must explicitly validate subject existence and subject-site consistency. Missing subjects, unknown subjects, and subject-site mismatches should remain separate DQ failure reasons because they represent different remediation paths.

The overlap between these failure populations must be evaluated in the final combined DQ analysis before calculating the expected quarantine population.

## 11. Adverse-Event Lifecycle and Date Consistency

### Question

Are adverse-event onset, resolution, and outcome fields temporally and clinically consistent?

### Purpose

Evaluate whether adverse-event lifecycle information is internally coherent before defining Silver DQ rules.

The analysis checks for:

- missing onset dates,
- resolution dates preceding onset dates,
- resolved events without a resolution date,
- unresolved events carrying a resolution date,
- and the proportion of events that remain unresolved.

The unresolved population is also important when evaluating whether the currently observed append-only source behavior can safely be assumed to continue.

In [0]:
%sql

WITH ae AS (

    SELECT
        ae_id,

        TRY_TO_DATE(onset_date) AS onset_date,

        TRY_TO_DATE(resolution_date) AS resolution_date,

        UPPER(TRIM(outcome)) AS outcome,

        _source_file_name

    FROM clinical_trial_intelligence.bronze.safety_adverse_events
)

SELECT
    COUNT(*) AS total_rows,
    COUNT_IF(
        onset_date IS NULL
    ) AS missing_onset_date,

    COUNT_IF(
        resolution_date IS NOT NULL
        AND onset_date IS NOT NULL
        AND resolution_date < onset_date
    ) AS resolution_before_onset,

    COUNT_IF(
        outcome = 'RESOLVED'
        AND resolution_date IS NULL
    ) AS resolved_without_resolution_date,

    COUNT_IF(
        outcome <> 'RESOLVED'
        AND resolution_date IS NOT NULL
    ) AS nonresolved_with_resolution_date,

    COUNT_IF(
        onset_date IS NOT NULL
        AND resolution_date IS NULL
    ) AS open_events,

    COUNT(DISTINCT CASE
        WHEN onset_date IS NOT NULL
         AND resolution_date IS NULL
        THEN ae_id
    END) AS distinct_open_events,

    COUNT(DISTINCT CASE
        WHEN onset_date IS NOT NULL
         AND resolution_date IS NULL
        THEN _source_file_name
    END) AS open_event_source_files

FROM ae;

## 12. Outcome Domain and Resolution-Date Consistency

### Question

What outcome values occur in the adverse-event feed, and how do they relate to the presence or absence of a resolution date?

### Purpose

Profile the actual adverse-event outcome domain before defining lifecycle consistency rules.

This prevents Silver logic from assuming that only one outcome value represents a completed adverse-event lifecycle and determines which outcome states legitimately require, permit, or prohibit a resolution date.

In [0]:

%sql

SELECT
    COALESCE(
        UPPER(TRIM(outcome)),
        '<NULL>'
    ) AS outcome,

    COUNT(*) AS row_count,

    COUNT_IF(
        resolution_date IS NULL
        OR TRIM(resolution_date) IN ('', '-')
    ) AS without_resolution_date,

    COUNT_IF(
        resolution_date IS NOT NULL
        AND TRIM(resolution_date) NOT IN ('', '-')
    ) AS with_resolution_date,

    COUNT(DISTINCT _source_file_name)
        AS source_file_count

FROM clinical_trial_intelligence.bronze.safety_adverse_events

GROUP BY
    COALESCE(
        UPPER(TRIM(outcome)),
        '<NULL>'
    )

ORDER BY row_count DESC;

### Result

Three standardized outcome states are present in the Bronze adverse-events feed:

- `RECOVERED`: 1,144 records, all with a populated `resolution_date`
- `ONGOING`: 427 records, all without a `resolution_date`
- `RECOVERING`: 381 records, all without a `resolution_date`

The relationship between adverse-event outcome and resolution-date availability is fully consistent across the observed data.

All recovered events contain a resolution date, while all ongoing and recovering events remain without one.

### Conclusion

The earlier apparent population of 1,144 non-resolved events with a resolution date was caused by testing for the value `RESOLVED`, while the actual source outcome domain uses `RECOVERED`.

The observed outcome domain is internally consistent with adverse-event lifecycle dates.

Silver lifecycle validation should therefore treat `RECOVERED` as the completed outcome state requiring a valid `resolution_date`, while `ONGOING` and `RECOVERING` should normally have no resolution date.

The outcome values themselves require no normalization based on the currently observed source data.

## 13. Causality and Action-Taken Domain Profile

### Question

What values occur in `related_to_study_drug` and `action_taken`, and are these domains consistent across the adverse-event feed?

### Purpose

Profile treatment-causality and action-taken values before defining Silver standardization and data-quality rules.

`related_to_study_drug` represents the reported relationship between an adverse event and study treatment, while `action_taken` records the treatment response associated with the event.

Profiling these domains determines whether normalization, reference-driven mapping, or additional DQ handling is required before the fields are used in downstream safety analysis.

In [0]:
%sql

SELECT
    'related_to_study_drug' AS field_name,
    COALESCE(UPPER(TRIM(related_to_study_drug)), '<NULL>') AS raw_value,
    COUNT(*) AS row_count,
    COUNT(DISTINCT ae_id) AS distinct_ae_ids,
    COUNT(DISTINCT _source_file_name) AS source_file_count

FROM clinical_trial_intelligence.bronze.safety_adverse_events

GROUP BY
    COALESCE(UPPER(TRIM(related_to_study_drug)), '<NULL>')

UNION ALL

SELECT
    'action_taken' AS field_name,
    COALESCE(UPPER(TRIM(action_taken)), '<NULL>') AS raw_value,
    COUNT(*) AS row_count,
    COUNT(DISTINCT ae_id) AS distinct_ae_ids,
    COUNT(DISTINCT _source_file_name) AS source_file_count

FROM clinical_trial_intelligence.bronze.safety_adverse_events

GROUP BY
    COALESCE(UPPER(TRIM(action_taken)), '<NULL>')

ORDER BY
    field_name,
    row_count DESC;

### Result

The adverse-event feed contains three observed causality values:

- `POSSIBLY RELATED`: 657 records
- `RELATED`: 648 records
- `NOT RELATED`: 647 records

The `action_taken` field contains four observed values:

- `DRUG INTERRUPTED`: 523 records
- `NONE`: 489 records
- `DOSE REDUCED`: 486 records
- `DRUG WITHDRAWN`: 454 records

All observed values are populated, consistently formatted after trimming and case normalization, and represented across all 10 source files.

No additional raw variants or missing values were identified in either domain.

### Conclusion

The observed `related_to_study_drug` and `action_taken` domains are structurally clean and do not currently require corrective normalization beyond standard trimming and case standardization.

The three causality categories should be preserved as distinct clinical values rather than collapsed into a binary related/not-related indicator, since `POSSIBLY RELATED` carries different information from both `RELATED` and `NOT RELATED`.

Similarly, the four observed action categories should remain distinct for downstream safety analysis.

Because these are controlled clinical domains, Silver should validate their accepted values so that unexpected future source values are detected rather than silently propagated.

## 14. Severity Mapping and Unmapped Grade Investigation

### Question

Which raw adverse-event severity values are present, which values resolve through the severity reference mapping, and what is the clinical profile of unmapped values?

### Purpose

Validate severity standardization before implementing Silver transformation logic.

This analysis quantifies each raw severity value, determines whether it is covered by the severity reference mapping, and examines seriousness and outcome patterns for unmapped values.

The objective is to avoid silently converting, discarding, or nullifying clinically meaningful severity information without supporting evidence.

In [0]:
%sql

WITH severity_ref AS (

    SELECT DISTINCT
        UPPER(TRIM(raw_severity)) AS raw_severity,
        UPPER(TRIM(standard_severity)) AS standard_severity

    FROM clinical_trial_intelligence.bronze.ref_severity_mapping
),

ae AS (

    SELECT
        ae_id,
        COALESCE(UPPER(TRIM(severity)), '<NULL>') AS raw_severity,
        UPPER(TRIM(serious_flag)) AS serious_flag,
        UPPER(TRIM(outcome)) AS outcome,
        _source_file_name

    FROM clinical_trial_intelligence.bronze.safety_adverse_events
)

SELECT
    ae.raw_severity,

    r.standard_severity,

    CASE
        WHEN r.raw_severity IS NULL
        THEN 'UNMAPPED'
        ELSE 'MAPPED'
    END AS mapping_status,

    COUNT(*) AS row_count,

    COUNT(DISTINCT ae.ae_id) AS distinct_ae_ids,

    COUNT_IF(ae.serious_flag IN ('Y', 'YES', '1', 'TRUE'))
        AS serious_rows,

    COUNT_IF(ae.serious_flag IN ('N', 'NO', '0', 'FALSE'))
        AS non_serious_rows,

    COUNT_IF(ae.outcome = 'RECOVERED')
        AS recovered_rows,

    COUNT_IF(ae.outcome = 'ONGOING')
        AS ongoing_rows,

    COUNT_IF(ae.outcome = 'RECOVERING')
        AS recovering_rows,

    COUNT(DISTINCT ae._source_file_name)
        AS source_file_count

FROM ae

LEFT JOIN severity_ref r
    ON ae.raw_severity = r.raw_severity

GROUP BY
    ae.raw_severity,
    r.standard_severity,
    CASE
        WHEN r.raw_severity IS NULL THEN 'UNMAPPED'
        ELSE 'MAPPED'
    END

ORDER BY
    mapping_status DESC,
    row_count DESC;

### Result

The adverse-event severity domain contains both directly standardized values and grade-based variants.

Mapped values include:

- `MILD` → `MILD`
- `1` → `MILD`
- `GRADE 1` → `MILD`
- `MODERATE` → `MODERATE`
- `2` → `MODERATE`
- `GRADE 2` → `MODERATE`
- `SEVERE` → `SEVERE`
- `3` → `SEVERE`
- `GRADE 3` → `SEVERE`

The following values are currently unmapped:

- `<NULL>`: 6 records
- `GRADE 4`: 4 records
- `UNKNOWN`: 4 records
- `N/A`: 3 records

`GRADE 4` is particularly important because it represents an explicit graded severity value rather than missing or indeterminate information.

The four `GRADE 4` records are currently marked non-serious in the source, indicating a potentially important disagreement between severity grade and the source-provided seriousness classification.

The remaining unmapped values (`UNKNOWN`, `N/A`, and NULL) represent missing or indeterminate severity information rather than a known severity grade.

### Conclusion

The severity reference mapping is incomplete and should be corrected before the adverse-events Silver pipeline is finalized.

`GRADE 4` must not be treated as an ordinary unmapped value or silently converted to NULL, because it contains clinically meaningful severity information.

However, it should also not be automatically collapsed into the existing `SEVERE` category without explicitly defining the intended severity model.

The safest design is to preserve the original source severity and introduce a separate normalized grade representation when grade-based values are present.

For the current Silver design:

- `MILD`, `MODERATE`, and `SEVERE` mappings remain valid.
- Grade 1–3 mappings remain supported by the existing reference.
- `GRADE 4` requires explicit reference-model extension before Silver finalization.
- `UNKNOWN`, `N/A`, and NULL should remain identifiable as missing or indeterminate severity conditions rather than being assigned a clinical severity level.

The observed `GRADE 4` / non-serious combinations should also be evaluated in the subsequent seriousness-consistency analysis rather than corrected implicitly.

## 15. Seriousness and Severity Cross-Field Consistency

### Question

Are severity, serious-event classification, and outcome values internally consistent?

### Purpose

Identify clinically significant cross-field combinations that may indicate source inconsistency or require explicit Silver quality flags.

This analysis focuses on combinations such as:

- severe or high-grade events marked non-serious;
- serious events carrying mild severity;
- fatal outcomes marked non-serious;
- and high-grade events that may require heightened review.

The objective is to distinguish records that are structurally invalid from records that should be retained but flagged for safety review.

In [0]:
%sql

WITH ae AS (

    SELECT
        ae_id,
        UPPER(TRIM(severity)) AS raw_severity,
        UPPER(TRIM(serious_flag)) AS serious_flag,
        UPPER(TRIM(outcome)) AS outcome

    FROM clinical_trial_intelligence.bronze.safety_adverse_events
)

SELECT
    COUNT(*) AS total_rows,

    COUNT_IF(
        raw_severity IN ('SEVERE', '3', 'GRADE 3', 'GRADE 4', 'GRADE 5')
        AND serious_flag IN ('N', 'NO', '0', 'FALSE')
    ) AS high_severity_nonserious_rows,

    COUNT_IF(
        raw_severity IN ('MILD', '1', 'GRADE 1')
        AND serious_flag IN ('Y', 'YES', '1', 'TRUE')
    ) AS mild_serious_rows,

    COUNT_IF(
        outcome = 'FATAL'
        AND serious_flag IN ('N', 'NO', '0', 'FALSE')
    ) AS fatal_nonserious_rows,

    COUNT_IF(
        raw_severity IN ('GRADE 4', 'GRADE 5')
        AND serious_flag IN ('N', 'NO', '0', 'FALSE')
    ) AS grade4_5_nonserious_rows

FROM ae;

### Result

The cross-field consistency analysis identified several severity–seriousness combinations requiring clinical review.

Out of 1,952 adverse-event records:

- 118 records have high severity (`SEVERE`, Grade 3, Grade 4, or equivalent) while being classified as non-serious.
- 3 records have mild severity while being classified as serious.
- 0 records have a `FATAL` outcome while being classified as non-serious.
- All 4 observed `GRADE 4` records are classified as non-serious.

These findings demonstrate that severity and seriousness are not interchangeable fields in the source data. In particular, the four Grade 4/non-serious combinations require explicit review because Grade 4 represents a high clinical severity level while the source seriousness indicator remains negative.

### Result

The cross-field consistency analysis identified several severity–seriousness combinations requiring clinical review.

Out of 1,952 adverse-event records:

- 118 records have high severity (`SEVERE`, Grade 3, Grade 4, or equivalent) while being classified as non-serious.
- 3 records have mild severity while being classified as serious.
- 0 records have a `FATAL` outcome while being classified as non-serious.
- All 4 observed `GRADE 4` records are classified as non-serious.

These findings demonstrate that severity and seriousness are not interchangeable fields in the source data. In particular, the four Grade 4/non-serious combinations require explicit review because Grade 4 represents a high clinical severity level while the source seriousness indicator remains negative.

## 16. Adverse-Event History Across Source Files

### Question

Does the same `ae_id` appear more than once or across multiple source files?

### Purpose

Determine whether the adverse-event feed behaves as append-only in the currently observed data or whether existing adverse events are reissued with updated lifecycle information.

This is required before selecting the Silver ingestion strategy because adverse events may evolve from ongoing to recovered, may receive a resolution date later, or may have other clinically meaningful updates.

In [0]:
%sql

WITH ae_history AS (

    SELECT
        ae_id,
        COUNT(*) AS row_count,
        COUNT(DISTINCT _source_file_name) AS source_file_count,
        MIN(_source_file_modification_ts) AS first_seen,
        MAX(_source_file_modification_ts) AS last_seen

    FROM clinical_trial_intelligence.bronze.safety_adverse_events

    WHERE ae_id IS NOT NULL
      AND TRIM(ae_id) <> ''

    GROUP BY ae_id
)

SELECT
    COUNT(*) AS distinct_ae_ids,

    COUNT_IF(row_count > 1)
        AS repeated_ae_ids,

    COUNT_IF(source_file_count > 1)
        AS ae_ids_across_multiple_files,

    MAX(row_count)
        AS max_rows_per_ae,

    MAX(source_file_count)
        AS max_source_files_per_ae

FROM ae_history;

### Result

The Bronze adverse-event dataset contains 1,952 distinct `ae_id` values across 1,952 records.

No repeated adverse-event identifiers were observed:

- repeated `ae_id` values: **0**
- `ae_id` values appearing across multiple source files: **0**
- maximum rows per `ae_id`: **1**
- maximum source files per `ae_id`: **1**

Therefore, the currently available source files exhibit an append-only pattern at the `ae_id` grain.

However, the lifecycle analysis identified **795 currently open adverse events**. These events may subsequently receive updated outcome, resolution-date, severity, seriousness, or other lifecycle information in future source deliveries.

### Conclusion

The currently observed adverse-event feed behaves as append-only because every `ae_id` occurs exactly once and no identifier is reissued across the available source files.

This observation should not be interpreted as a permanent source-system guarantee. Adverse events are lifecycle entities, and the presence of 795 open events creates the possibility that existing `ae_id` values may be reissued when their clinical status changes.

Therefore, the Silver implementation should:

- enforce uniqueness of `ae_id`;
- detect repeated identifiers in future ingestion;
- avoid silently creating duplicate adverse events;
- preserve sufficient ingestion metadata to identify later source versions;
- treat the current append-only behavior as an observed ingestion characteristic rather than an immutable business rule.

If future files begin reissuing existing `ae_id` values with changed attributes, the ingestion strategy should transition to key-based CDC/upsert processing.

## 17. Combined Data-Quality Baseline

### Question

Based on the exploration findings, how many Bronze adverse-event records are expected to pass into Silver and how many should be quarantined?

### Purpose

Establish a reproducible pre-implementation baseline for the Silver adverse-event pipeline by combining all blocking data-quality conditions identified during exploration.

The baseline is used to verify that the implemented Silver transformation produces the expected valid and quarantine populations and to identify records affected by multiple blocking failures.

In [0]:
%sql

WITH subjects AS (
    SELECT DISTINCT
        subject_id,
        study_id,
        site_id
    FROM clinical_trial_intelligence.bronze.edc_subjects
),

sites AS (
    SELECT DISTINCT
        site_id,
        study_id
    FROM clinical_trial_intelligence.bronze.ctms_sites
),

ae_base AS (
    SELECT
        ae.*,

        TO_DATE(ae.onset_date, 'dd-MMM-yyyy') AS onset_dt,

        CASE
            WHEN ae.ae_id IS NULL
                 OR TRIM(ae.ae_id) = ''
            THEN 1 ELSE 0
        END AS f_missing_ae_id,

        CASE
            WHEN ae.subject_id IS NULL
                 OR TRIM(ae.subject_id) = ''
            THEN 1 ELSE 0
        END AS f_missing_subject_id,

        CASE
            WHEN ae.onset_date IS NULL
                 OR TRIM(ae.onset_date) = ''
                 OR TO_DATE(ae.onset_date, 'dd-MMM-yyyy') IS NULL
            THEN 1 ELSE 0
        END AS f_invalid_onset_date

    FROM clinical_trial_intelligence.bronze.safety_adverse_events ae
),

evaluated AS (
    SELECT
        a.*,

        CASE
            WHEN a.subject_id IS NOT NULL
             AND TRIM(a.subject_id) <> ''
             AND s.subject_id IS NULL
            THEN 1 ELSE 0
        END AS f_unknown_subject,

        CASE
            WHEN s.subject_id IS NOT NULL
             AND (
                    a.study_id <> s.study_id
                    OR a.site_id <> s.site_id
                 )
            THEN 1 ELSE 0
        END AS f_subject_relationship_mismatch,

        CASE
            WHEN st.site_id IS NULL
            THEN 1 ELSE 0
        END AS f_unknown_site,

        CASE
            WHEN st.site_id IS NOT NULL
             AND a.study_id <> st.study_id
            THEN 1 ELSE 0
        END AS f_site_study_mismatch

    FROM ae_base a

    LEFT JOIN subjects s
        ON a.subject_id = s.subject_id
       AND a.study_id = s.study_id
       AND a.site_id = s.site_id

    LEFT JOIN sites st
        ON a.site_id = st.site_id
       AND a.study_id = st.study_id
),

scored AS (
    SELECT
        *,

        (
            f_missing_ae_id
            + f_missing_subject_id
            + f_invalid_onset_date
            + f_unknown_subject
            + f_subject_relationship_mismatch
            + f_unknown_site
            + f_site_study_mismatch
        ) AS failure_count

    FROM evaluated
)

SELECT
    COUNT(*) AS total_rows,

    COUNT_IF(failure_count = 0)
        AS expected_silver_rows,

    COUNT_IF(failure_count > 0)
        AS expected_quarantine_rows,

    COUNT_IF(failure_count > 1)
        AS rows_with_multiple_failures,

    MAX(failure_count)
        AS max_failures_per_row

FROM scored;

### Result

The combined data-quality baseline evaluated all 1,952 Bronze adverse-event records against the blocking conditions identified during exploration.

The results are:

- Total Bronze records: **1,952**
- Expected Silver records: **1,879**
- Expected quarantine records: **73**
- Records with multiple blocking failures: **0**
- Maximum blocking failures per record: **1**

The baseline reconciles completely:

**1,879 Silver + 73 Quarantine = 1,952 Bronze records.**

No quarantined record currently violates more than one blocking rule.

### Design Conclusion

The adverse-event Silver pipeline is expected to retain **1,879 clinically usable records** and quarantine **73 records** that violate at least one blocking data-quality condition.

Blocking conditions are restricted to structural and referential-integrity failures that prevent reliable downstream use. Clinical consistency observations, such as severity–seriousness disagreements, remain warning-level indicators and should not independently cause quarantine.

The Silver implementation should therefore:

- standardize severity using the severity reference mapping;
- standardize the serious indicator;
- parse clinical dates into typed date columns;
- validate subject, study, and site relationships;
- route records failing blocking conditions to quarantine;
- retain warning-level clinical consistency indicators for downstream review;
- preserve source and ingestion metadata for traceability;
- enforce the `ae_id` grain and monitor future duplicate/reissued identifiers.

Based on the currently observed source files, `ae_id` behaves as an append-only business key. This remains an observed source characteristic rather than a permanent source-system guarantee.

In [0]:
%sql

SELECT *
FROM clinical_trial_intelligence.bronze.ref_severity_mapping
ORDER BY raw_severity;

In [0]:
%sql

SELECT
    raw_severity,
    standard_severity,
    severity_rank
FROM clinical_trial_intelligence.bronze.ref_severity_mapping
WHERE UPPER(TRIM(raw_severity)) IN ('GRADE 4', 'GRADE 5')
ORDER BY severity_rank;